In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

In [0]:


# -------------------------
# 1. Base identifiers
# -------------------------
order_ids = [f"O{i}" for i in range(1, 50001)]
customer_ids = [f"C{i}" for i in range(1, 50001)]

event_types = ["CREATED", "UPDATED", "COMPLETED", "CANCELLED"]
source_systems = ["web", "mobile", "api"]

base_time = datetime(2024, 1, 1, 9, 0, 0)

rows = []

# -------------------------
# 2. Generate dirty events
# -------------------------
for i in range(len(order_ids)):
    order_id = order_ids[i]
    customer_id = customer_ids[i]

    num_events = random.randint(1, 4)

    event_times = sorted(
        [base_time + timedelta(minutes=random.randint(0, 120)) for _ in range(num_events)]
    )

    for et in event_times:
        event_type = random.choice(event_types)

        order_amount = (
            None if random.random() < 0.15
            else round(random.uniform(100, 5000), 2)
        )

        ingestion_time = et + timedelta(minutes=random.randint(-3, 5))

        rows.append((
            order_id,
            customer_id,
            event_type,
            et,
            order_amount,
            random.choice(source_systems),
            ingestion_time
        ))

        # Introduce exact duplicate events
        if random.random() < 0.1:
            rows.append((
                order_id,
                customer_id,
                event_type,
                et,
                order_amount,
                random.choice(source_systems),
                ingestion_time
            ))

# -------------------------
# 3. Schema
# -------------------------
schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("order_amount", DoubleType(), True),
    StructField("source_system", StringType(), False),
    StructField("ingestion_time", TimestampType(), False)
])

# -------------------------
# 4. Create DataFrame
# -------------------------
orders_df = spark.createDataFrame(rows, schema)

display(orders_df.limit(10))


Write PySpark code to produce ONE FINAL RECORD PER order_id with the following rules:

✅ Business Rules

Deduplicate events

Exact duplicates may exist

Keep only one

Pick the latest valid state per order

Latest = by event_time

If event_time ties, use ingestion_time

Event precedence

If an order is CANCELLED, it should be the final state even if COMPLETED arrived later

Priority order:

CANCELLED > COMPLETED > UPDATED > CREATED


Order amount handling

Use the latest non-null order_amount

Even if the latest event has NULL

Output columns

order_id
customer_id
final_status
final_order_amount
final_event_time

In [0]:
display(orders_df.limit(10))